# 1. Data Loading

In [928]:
import pandas as pd

campaigns = pd.read_csv("/Users/laiaalcala/data_technical_tests/data/raw/raw_campaigns.csv")
clients = pd.read_csv("/Users/laiaalcala/data_technical_tests/data/raw/raw_clients.csv")
pos = pd.read_csv("/Users/laiaalcala/data_technical_tests/data/raw/raw_pos.csv")
projects = pd.read_csv("/Users/laiaalcala/data_technical_tests/data/raw/raw_projects.csv")
questions = pd.read_csv("/Users/laiaalcala/data_technical_tests/data/raw/raw_questions.csv")
responses = pd.read_csv("/Users/laiaalcala/data_technical_tests/data/raw/raw_responses.csv")
route_employee = pd.read_csv("/Users/laiaalcala/data_technical_tests/data/raw/raw_route_employee.csv")
routes = pd.read_csv("/Users/laiaalcala/data_technical_tests/data/raw/raw_routes.csv")
visits = pd.read_csv("/Users/laiaalcala/data_technical_tests/data/raw/raw_visits.csv")
workers = pd.read_csv("/Users/laiaalcala/data_technical_tests/data/raw/raw_workers.csv")

# 2. Data Exploration

In [929]:
#Function for data exploration
def exploration(df):
    print("First 10 rows of the table:")
    display(df.head(10))
    
    print("\nData types and non-null counts:")
    display(df.info())
    
    print("\nDescriptive statistics for all columns:")
    display(df.describe(include="all").T)
    
    print("\n% of NaNs per column:")
    print(df.isnull().mean().sort_values(ascending=False))

    print("\n% of Empty Strings ('') per column:")
    # Isolate only text/categorical columns to check for empty strings
    string_cols = df.select_dtypes(include=['object', 'string']).columns

    if len(string_cols) == 0:
        print("No text columns found to analyze empty strings.")
    else:
        # Evaluate cell by cell only on text columns: check if a string is empty after stripping spaces
        # Actual NaNs are filtered out via pd.notnull(x) to prevent double-counting
        empty_counts = df[string_cols].map(lambda x: str(x).strip() == "" if pd.notnull(x) else False)
        print(empty_counts.mean().sort_values(ascending=False))

    duplicate_count = df.duplicated().sum()
    duplicate_pct = df.duplicated().mean() * 100
    print(f"\n Total duplicate rows: {duplicate_count} ({duplicate_pct:.2f}%)")

    print("\n % Outliers Detection (IQR Method):")
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    if len(numeric_cols) == 0:
        print("No numeric columns found to analyze outliers.")
    else:
        outlier_summary = []
        for col in numeric_cols:
            # Temporarily drop null values to ensure accurate statistical calculations
            col_clean = df[col].dropna()
            if len(col_clean) > 0:
                Q1 = col_clean.quantile(0.25)
                Q3 = col_clean.quantile(0.75)
                IQR = Q3 - Q1
                # Filter and isolate rows that fall outside the calculated thresholds
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR
                
                # Filter and isolate rows that fall outside the calculated thresholds
                outliers = col_clean[(col_clean < lower_bound) | (col_clean > upper_bound)]
                # Append the calculated metrics and outlier metadata to the summary list
                outlier_summary.append({
                    "Column": col,
                    "Outliers count": len(outliers),
                    "% Outliers": f"{(len(outliers) / len(df)) * 100:.2f}%",
                    "Min outlier found": outliers.min() if len(outliers) > 0 else None,
                    "Max outlier found": outliers.max() if len(outliers) > 0 else None
                })
        
        display(pd.DataFrame(outlier_summary).set_index("Column"))

    #Validate dates and see date ranges
    date_cols = [col for col in df.columns if 'date' in col.lower() or col.lower().endswith('_at') or col.lower().endswith('_at_sys') or 'delete' in col.lower()] or df.select_dtypes(include=['datetime64']).columns
    if len(date_cols) == 0:
        print("No potential date columns detected.")
    else:
        date_summary = []
        for col in date_cols:
            # Safely try to convert to datetime (errors='coerce' turns invalid formats into NaT)
            temp_date = pd.to_datetime(df[col], errors='coerce')
            
            # If the conversion successfully yielded at least some valid dates
            if temp_date.notnull().sum() > 0:
                min_date = temp_date.min()
                max_date = temp_date.max()
                days_span = (max_date - min_date).days if pd.notnull(min_date) and pd.notnull(max_date) else None
                
                date_summary.append({
                    "Date Column": col,
                    "Min Date (Start)": min_date,
                    "Max Date (End)": max_date,
                    "Days Timespan": days_span,
                    "Valid Dates Count": temp_date.notnull().sum()
                })
        
        if len(date_summary) > 0:
            print('\n Date summary:')
            display(pd.DataFrame(date_summary).set_index("Date Column"))
        else:
            print("Columns with date-like names could not be parsed into valid date ranges.")


#### 2.1 Campaigns

In [930]:
#campaigns['campaign_id'] = campaigns['campaign_id'].astype(str)
#campaigns['campaign_state_id'] = campaigns['campaign_state_id'].astype(str)
#campaigns['project_id'] = campaigns['project_id'].astype(str)

exploration(campaigns)

First 10 rows of the table:


,campaign_id,campaign_code,campaign_name,project_id,project_code,client_code,campaign_start_date,campaign_end_date,is_active,campaign_state_id,campaign_state,total_visits_planned,total_pos_planned,visit_duration_minutes,created_at,updated_at_sys
0,1,CAM00001,Campaña Oct 2025 – Proyecto Mantenimiento 1,1,PRJ0001,CLI0001,20/10/2025,12/03/2026,False,5,Activa,141,52,45.0,2025-09-30,2025-12-13
1,2,CAM00002,Campaña Jan 2026 – Proyecto Mantenimiento 1,1,PRJ0001,CLI0001,2026-01-11,2026-02-25,False,4,En preparación,376,197,60.0,2025-12-15,2026-03-14
2,3,CAM00003,Campaña Nov 2025 – Proyecto Formación 2,2,PRJ0002,CLI0002,2025-11-18,2026-03-03,False,1,NaN,45,159,45.0,2025-10-28,2025-12-28
3,4,CAM00004,Campaña Oct 2025 – Proyecto Formación 2,2,PRJ0002,CLI0002,2025-10-15,2026-03-24,False,2,Pausada,311,73,60.0,2025-10-09,2026-03-22
4,5,CAM00005,Campaña Jan 2026 – Proyecto Visibilidad 3,3,PRJ0003,CLI0002,2026-01-16,2026-07-14,True,3,Cerrada,222,43,30.0,2026-01-01,2026-04-22
5,6,CAM00006,Campaña Oct 2025 – Proyecto Visibilidad 4,4,PRJ0004,CLI0002,2025-10-03,2026-02-27,False,5,Cerrada,155,43,NaN,2025-09-21,2026-01-05
6,7,CAM00007,Campaña Nov 2025 – Proyecto Auditoria 5,5,PRJ0005,CLI0003,2025-11-10,2026-04-01,False,5,Activa,303,86,15.0,2025-10-28,2025-12-09
7,8,CAM00008,Campaña Oct 2025 – Proyecto Auditoria 5,5,PRJ0005,CLI0003,2025-10-28,2026-04-17,False,3,Cerrada,344,228,30.0,2025-10-08,2025-12-31
8,9,CAM00009,Campaña Oct 2025 – Proyecto Auditoria 5,5,PRJ0005,CLI0003,2025-10-14,2025-12-06,NaN,3,Cerrada,154,51,45.0,2025-09-17,2026-01-31
9,10,CAM00010,Campaña Oct 2025 – Proyecto Mantenimiento 6,6,PRJ0006,CLI0004,2025-10-29,2025-12-17,False,1,En preparación,302,47,45.0,2025-10-23,2026-01-15



Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   campaign_id             72 non-null     int64  
 1   campaign_code           70 non-null     object 
 2   campaign_name           72 non-null     object 
 3   project_id              72 non-null     int64  
 4   project_code            65 non-null     object 
 5   client_code             72 non-null     object 
 6   campaign_start_date     72 non-null     object 
 7   campaign_end_date       72 non-null     object 
 8   is_active               67 non-null     object 
 9   campaign_state_id       72 non-null     int64  
 10  campaign_state          59 non-null     object 
 11  total_visits_planned    72 non-null     int64  
 12  total_pos_planned       72 non-null     int64  
 13  visit_duration_minutes  59 non-null     float64
 14  created_at 

None


Descriptive statistics for all columns:


...


% of NaNs per column:
campaign_state            0.180556
visit_duration_minutes    0.180556
project_code              0.097222
is_active                 0.069444
campaign_code             0.027778
campaign_id               0.000000
campaign_name             0.000000
project_id                0.000000
client_code               0.000000
campaign_start_date       0.000000
campaign_end_date         0.000000
campaign_state_id         0.000000
total_visits_planned      0.000000
total_pos_planned         0.000000
created_at                0.000000
updated_at_sys            0.000000
dtype: float64

% of Empty Strings ('') per column:
campaign_code          0.0
campaign_name          0.0
project_code           0.0
client_code            0.0
campaign_start_date    0.0
campaign_end_date      0.0
is_active              0.0
campaign_state         0.0
created_at             0.0
updated_at_sys         0.0
dtype: float64

 Total duplicate rows: 0 (0.00%)

 % Outliers Detection (IQR Method):


,Outliers count,% Outliers,Min outlier found,Max outlier found
Column,,,,
campaign_id,0,0.00%,None,None
project_id,0,0.00%,None,None
campaign_state_id,0,0.00%,None,None
total_visits_planned,0,0.00%,None,None
total_pos_planned,0,0.00%,None,None
visit_duration_minutes,0,0.00%,None,None



 Date summary:


/var/folders/gv/36w305r13j7flz95rx8pfq2h0000gn/T/ipykernel_34756/512316687.py:69: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  temp_date = pd.to_datetime(df[col], errors='coerce')


,Min Date (Start),Max Date (End),Days Timespan,Valid Dates Count
Date Column,,,,
campaign_start_date,2025-10-20,2026-02-17,120,12
campaign_end_date,2026-01-03,2026-12-03,334,4
created_at,2025-09-11,2026-02-13,155,72
updated_at_sys,2025-11-14,2026-05-01,168,72


In [931]:
print(campaigns['created_at'].unique())
print(campaigns['updated_at_sys'].unique())
print(campaigns['is_active'].unique())

['2025-09-30' '2025-12-15' '2025-10-28' '2025-10-09' '2026-01-01'
 '2025-09-21' '2025-10-08' '2025-09-17' '2025-10-23' '2025-09-11'
 '2025-12-25' '2025-11-08' '2026-01-16' '2026-01-02' '2025-12-29'
 '2026-01-21' '2025-11-26' '2025-10-10' '2025-10-14' '2025-11-21'
 '2025-12-08' '2025-12-21' '2025-12-18' '2025-09-26' '2026-01-20'
 '2025-10-01' '2026-02-04' '2025-10-17' '2025-12-02' '2025-11-04'
 '2026-01-08' '2026-01-19' '2025-10-07' '2026-02-13' '2026-01-13'
 '2025-09-25' '2026-01-30' '2026-02-05' '2025-12-04' '2025-10-04'
 '2026-02-06' '2026-01-27' '2025-11-27' '2025-11-25' '2026-01-07'
 '2025-12-16' '2025-11-28' '2026-02-08' '2025-11-15' '2025-12-24'
 '2025-10-24' '2026-01-12' '2025-12-20' '2025-11-05' '2025-09-28']
['2025-12-13' '2026-03-14' '2025-12-28' '2026-03-22' '2026-04-22'
 '2026-01-05' '2025-12-09' '2025-12-31' '2026-01-31' '2026-01-15'
 '2025-11-25' '2026-02-11' '2026-02-18' '2026-03-27' '2026-01-28'
 '2026-04-26' '2026-02-27' '2026-02-06' '2026-03-19' '2026-02-23'
 '2026-01

Campaign ID is unique.  
We can see that the format of some dates are not homogeneous in the same column.  
is_active column has 'nan' values.

__1 Campaign can have N Projects__

#### 2.2 Clients

In [932]:
#clients['client_id'] = clients['client_id'].astype(str)
#clients['company_id'] = clients['company_id'].astype(str)

exploration(clients)

First 10 rows of the table:


,client_id,client_name,company_id,country,sector,created_at,updated_at_sys
0,1,Laboratorios Valtera España,1,spain,Farma,2021-10-17,2023-06-13
1,2,Pharmakon Consumer Health,1,España,Farma,2020-05-14,2021-08-04
2,3,Medicolab Iberia,1,spain,Farma,2019-04-22,2022-06-14
3,4,Biofarma España SL,1,spain,Gran Consumo,2019-02-12,2020-05-08
4,5,Salutis Consumer España,1,España,Farma,2020-07-08,2021-05-30
5,6,Farmanat Group,1,España,Farma,2021-11-30,2022-05-08
6,7,Dermocare Iberia,1,España,Farma,2019-03-17,2019-09-22
7,8,Novapharma España,1,España,Farma,2020-02-18,2023-07-08
8,9,Aquilea Consumer Health,1,SPAIN,Parafarmacia,2019-06-26,2020-03-07
9,10,Clinipharma España,1,spain,Farma,2020-02-15,2023-08-25



Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   client_id       15 non-null     int64 
 1   client_name     15 non-null     object
 2   company_id      15 non-null     int64 
 3   country         15 non-null     object
 4   sector          15 non-null     object
 5   created_at      15 non-null     object
 6   updated_at_sys  15 non-null     object
dtypes: int64(2), object(5)
memory usage: 968.0+ bytes


None


Descriptive statistics for all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
client_id,15.0,NaN,NaN,NaN,7.066667,4.366539,1.0,3.5,7.0,10.5,14.0
client_name,15,14,Laboratorios Valtera España,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
company_id,15.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
country,15,3,España,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sector,15,3,Farma,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
created_at,15,13,2021-10-17,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
updated_at_sys,15,14,2023-06-13,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN



% of NaNs per column:
client_id         0.0
client_name       0.0
company_id        0.0
country           0.0
sector            0.0
created_at        0.0
updated_at_sys    0.0
dtype: float64

% of Empty Strings ('') per column:
client_name       0.0
country           0.0
sector            0.0
created_at        0.0
updated_at_sys    0.0
dtype: float64

 Total duplicate rows: 1 (6.67%)

 % Outliers Detection (IQR Method):


,Outliers count,% Outliers,Min outlier found,Max outlier found
Column,,,,
client_id,0,0.00%,None,None
company_id,0,0.00%,None,None



 Date summary:


,Min Date (Start),Max Date (End),Days Timespan,Valid Dates Count
Date Column,,,,
created_at,2018-05-23,2021-11-30,1287,15
updated_at_sys,2019-04-14,2023-08-25,1594,15


In [933]:
# Sort by employee ID so we can see the duplicates grouped one under the other
# keep=False marks all occurrences of the duplicate as True
display(clients[clients.duplicated(keep=False)].sort_values(by='client_id'))

,client_id,client_name,company_id,country,sector,created_at,updated_at_sys
0,1,Laboratorios Valtera España,1,spain,Farma,2021-10-17,2023-06-13
14,1,Laboratorios Valtera España,1,spain,Farma,2021-10-17,2023-06-13


In [934]:
print(clients['country'].unique())

['spain' 'España' 'SPAIN']


In [935]:
print(clients['created_at'].unique())
print(clients['updated_at_sys'].unique())

['2021-10-17' '2020-05-14' '2019-04-22' '2019-02-12' '2020-07-08'
 '2021-11-30' '2019-03-17' '2020-02-18' '2019-06-26' '2020-02-15'
 '2021-06-20' '2018-05-23' '2021-07-25']
['2023-06-13' '2021-08-04' '2022-06-14' '2020-05-08' '2021-05-30'
 '2022-05-08' '2019-09-22' '2023-07-08' '2020-03-07' '2023-08-25'
 '2023-06-11' '2019-04-14' '2021-11-05' '2022-07-23']


There are no nulls nor empty spaces in the table of Clients. 
Client ID and Client name are unique without taking into account the duplicate.     
Categorical inconsistency detected in the country field. The column contains mixed languages (English/Spanish) and inconsistent casing.

#### 2.3 Points of sale / Intervention points

In [936]:
#pos['intervention_point_id'] = pos['intervention_point_id'].astype(str)

exploration(pos)

First 10 rows of the table:


,intervention_point_id,intervention_point_code,intervention_point_name,intervention_point_address,intervention_point_province,intervention_point_locality,intervention_point_postal_code,intervention_point_latitude,intervention_point_longitude,intervention_point_is_active,created_at,updated_at
0,1,PDV000001,CONDISLIFE EL TEU SÚPER 1,Calle 20 Nº 41,GUADALAJARA,Localidad 50,46365.0,41.774104,-5.411961,True,2020-05-07,2025-01-26
1,2,PDV000002,CONDIS EL TEU SÚPER 2,Calle 68 Nº 12,"Palmas, Las",Localidad 32,40996.0,37.338316,-7.198834,False,2023-06-11,2025-10-20
2,3,PDV000003,CEPSA 3,Calle 27 Nº 3,Asturias,Localidad 37,17423.0,39.344115,-6.713749,True,2023-09-23,2025-02-03
3,4,PDV000004,CEPSA 4,NaN,Évora,Localidad 46,33157.0,39.270433,0.871244,NaN,2022-02-26,2025-12-06
4,5,PDV000005,DIA % MARKET 5,Calle 146 Nº 35,CÁCERES,Localidad 44,NaN,40.742044,-7.411996,True,2022-07-30,2026-02-17
5,6,PDV000006,MAIS PERTO 6,Calle 46 Nº 36,Castelló,Localidad 43,20586.0,38.925785,-0.932595,False,2023-07-19,2025-03-24
6,7,PDV000007,EL CORTE INGLÉS 7,Calle 50 Nº 2,ZAMORA,Localidad 16,NaN,41.945544,-4.807229,NaN,2020-03-16,2024-12-13
7,8,PDV000008,CONTINENTE MODELO 8,Calle 11 Nº 26,Barcelona,Localidad 42,35283.0,43.050673,-6.712556,True,2020-03-16,2026-01-23
8,9,PDV000009,NaN,Calle 121 Nº 40,Lleida,Localidad 45,NaN,37.291868,2.355948,False,2020-12-15,2025-08-28
9,10,PDV000010,CAPRABO 10,Calle 153 Nº 40,Pontevedra,Localidad 13,23108.0,41.087984,-7.050371,True,2021-03-12,2026-02-21



Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   intervention_point_id           250 non-null    int64  
 1   intervention_point_code         236 non-null    object 
 2   intervention_point_name         244 non-null    object 
 3   intervention_point_address      235 non-null    object 
 4   intervention_point_province     236 non-null    object 
 5   intervention_point_locality     224 non-null    object 
 6   intervention_point_postal_code  224 non-null    float64
 7   intervention_point_latitude     235 non-null    float64
 8   intervention_point_longitude    243 non-null    float64
 9   intervention_point_is_active    240 non-null    object 
 10  created_at                      250 non-null    object 
 11  updated_at                      250 non-null    object 
dtypes: 

None


Descriptive statistics for all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
intervention_point_id,250.0,NaN,NaN,NaN,125.5,72.312977,1.0,63.25,125.5,187.75,250.0
intervention_point_code,236,236,PDV000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
intervention_point_name,244,244,CONDISLIFE EL TEU SÚPER 1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
intervention_point_address,235,234,Calle 176 Nº 39,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
intervention_point_province,236,59,Castellon,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
intervention_point_locality,224,50,Localidad 48,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
intervention_point_postal_code,224.0,NaN,NaN,NaN,27124.022321,15543.758634,1270.0,11935.5,27470.0,41000.0,52814.0
intervention_point_latitude,235.0,NaN,NaN,NaN,39.583861,2.162975,36.048152,37.56009,39.4659,41.359054,43.371965
intervention_point_longitude,243.0,NaN,NaN,NaN,-2.411287,3.50584,-8.474971,-5.714012,-2.499989,0.729985,3.395517
intervention_point_is_active,240,2,True,170,NaN,NaN,NaN,NaN,NaN,NaN,NaN



% of NaNs per column:
intervention_point_locality       0.104
intervention_point_postal_code    0.104
intervention_point_address        0.060
intervention_point_latitude       0.060
intervention_point_code           0.056
intervention_point_province       0.056
intervention_point_is_active      0.040
intervention_point_longitude      0.028
intervention_point_name           0.024
intervention_point_id             0.000
created_at                        0.000
updated_at                        0.000
dtype: float64

% of Empty Strings ('') per column:
intervention_point_code         0.0
intervention_point_name         0.0
intervention_point_address      0.0
intervention_point_province     0.0
intervention_point_locality     0.0
intervention_point_is_active    0.0
created_at                      0.0
updated_at                      0.0
dtype: float64

 Total duplicate rows: 0 (0.00%)

 % Outliers Detection (IQR Method):


,Outliers count,% Outliers,Min outlier found,Max outlier found
Column,,,,
intervention_point_id,0,0.00%,None,None
intervention_point_postal_code,0,0.00%,None,None
intervention_point_latitude,0,0.00%,None,None
intervention_point_longitude,0,0.00%,None,None



 Date summary:


,Min Date (Start),Max Date (End),Days Timespan,Valid Dates Count
Date Column,,,,
created_at,2020-01-17,2024-01-01,1445,250
updated_at,2024-01-04,2026-04-29,846,250


In [937]:
print(pos['intervention_point_is_active'].unique())
print(pos['intervention_point_province'].unique())

[True False nan]
['GUADALAJARA' 'Palmas, Las' 'Asturias' 'Évora' 'CÁCERES' 'Castelló'
 'ZAMORA' 'Barcelona' 'Lleida' 'Pontevedra' 'Valencia/València'
 'Viseu Dão Lafões' 'Zamora' 'Almería' 'Valladolid' 'Segovia' 'Girona'
 'A Coruña' 'Cáceres' 'Andorra la Vella' 'Santarém' nan 'Palencia'
 'VALLADOLID' 'CIUDAD REAL' 'Zaragoza' 'Albacete' 'La Rioja' 'TARRAGONA'
 'Escaldes-Engordany' 'MADRID' 'Portalegre' 'SANTA CRUZ DE TENERIFE'
 'Canillo' 'Cantabria' 'Guarda' 'Ávila' 'BADAJOZ' 'HUELVA' 'Madrid'
 'Madeira' 'Castellon' 'Viseu' 'Setúbal' 'Cuenca' 'Granada' 'ZARAGOZA'
 'Coimbra' 'Hauts-de-Seine' 'MÁLAGA' 'ASTURIAS' 'Rioja, La' 'Tarragona'
 'PONTEVEDRA' 'LUGO' 'GIRONA' 'Castellón' 'LLEIDA' 'Alpes (Hautes)' 'Beja']


In [938]:
#print(pos['created_at'].unique())
#print(pos['updated_at'].unique())

The `intervention_point_id` is unique.  
The `intervention_point_is_active` field contains structural nulls (`nan`), while the `intervention_point_province` column suffers from a high degree of text dispersion. This includes inconsistent casing (e.g., `ZAMORA` vs. `Zamora`), overlapping characters (`Castellón` vs. `Castellon`), and word order inversions (`La Rioja` vs. `Rioja, La`).

#### 2.4 Projects

In [939]:
#projects['project_id'] = projects['project_id'].astype(str)
#projects['client_id'] = projects['client_id'].astype(str)

exploration(projects)

First 10 rows of the table:


,project_id,project_name,project_code,client_id,client_code,created_at,updated_at_sys,project_exportable
0,1,Proyecto Mantenimiento 1,PRJ0001,1,CLI0001,2024-09-30,2024-11-05,False
1,2,Proyecto Formación 2,PRJ0002,2,CLI0002,2024-09-07,2025-01-23,False
2,3,Proyecto Visibilidad 3,PRJ0003,2,CLI0002,2024-11-28,2025-05-08,True
3,4,Proyecto Visibilidad 4,PRJ0004,2,CLI0002,2024-10-01,2025-12-23,True
4,5,Proyecto Auditoria 5,PRJ0005,3,CLI0003,2025-05-26,2025-07-01,True
5,6,Proyecto Mantenimiento 6,PRJ0006,4,CLI0004,2024-09-09,2025-01-21,True
6,7,Proyecto Sell-In 7,PRJ0007,5,CLI0005,2025-02-12,2025-10-30,True
7,8,Proyecto Visibilidad 8,PRJ0008,5,CLI0005,2024-04-03,2024-09-13,True
8,9,Proyecto Auditoria 9,PRJ0009,5,CLI0005,2024-03-06,2025-06-28,True
9,10,Proyecto Mantenimiento 10,PRJ0010,6,CLI0006,2024-01-12,2025-12-10,True



Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   project_id          28 non-null     int64 
 1   project_name        28 non-null     object
 2   project_code        26 non-null     object
 3   client_id           28 non-null     int64 
 4   client_code         28 non-null     object
 5   created_at          28 non-null     object
 6   updated_at_sys      28 non-null     object
 7   project_exportable  28 non-null     bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 1.7+ KB


None


Descriptive statistics for all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
project_id,28.0,NaN,NaN,NaN,14.5,8.225975,1.0,7.75,14.5,21.25,28.0
project_name,28,28,Proyecto Mantenimiento 1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
project_code,26,26,PRJ0001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
client_id,28.0,NaN,NaN,NaN,7.357143,4.102522,1.0,4.75,7.0,11.25,14.0
client_code,28,14,CLI0002,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
created_at,28,27,2025-02-12,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
updated_at_sys,28,27,2025-06-16,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
project_exportable,28,2,True,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN



% of NaNs per column:
project_code          0.071429
project_id            0.000000
project_name          0.000000
client_id             0.000000
client_code           0.000000
created_at            0.000000
updated_at_sys        0.000000
project_exportable    0.000000
dtype: float64

% of Empty Strings ('') per column:
project_name      0.0
project_code      0.0
client_code       0.0
created_at        0.0
updated_at_sys    0.0
dtype: float64

 Total duplicate rows: 0 (0.00%)

 % Outliers Detection (IQR Method):


,Outliers count,% Outliers,Min outlier found,Max outlier found
Column,,,,
project_id,0,0.00%,None,None
client_id,0,0.00%,None,None



 Date summary:


,Min Date (Start),Max Date (End),Days Timespan,Valid Dates Count
Date Column,,,,
created_at,2024-01-04,2025-05-26,508,28
updated_at_sys,2024-05-26,2025-12-23,576,28


In [940]:
print(projects['created_at'].unique())
print(projects['updated_at_sys'].unique())
print(projects['project_exportable'].unique())

['2024-09-30' '2024-09-07' '2024-11-28' '2024-10-01' '2025-05-26'
 '2024-09-09' '2025-02-12' '2024-04-03' '2024-03-06' '2024-01-12'
 '2024-12-14' '2024-01-04' '2024-04-18' '2024-06-05' '2024-11-27'
 '2024-02-29' '2024-05-08' '2025-03-09' '2024-09-10' '2024-08-13'
 '2024-08-22' '2024-10-12' '2025-04-29' '2024-04-09' '2025-04-23'
 '2025-02-16' '2025-04-04']
['2024-11-05' '2025-01-23' '2025-05-08' '2025-12-23' '2025-07-01'
 '2025-01-21' '2025-10-30' '2024-09-13' '2025-06-28' '2025-12-10'
 '2025-03-04' '2025-05-30' '2025-09-19' '2025-12-09' '2025-05-31'
 '2024-05-26' '2025-11-21' '2025-06-19' '2025-09-24' '2024-09-20'
 '2025-07-02' '2024-09-27' '2025-07-15' '2025-06-16' '2025-10-07'
 '2025-06-23' '2025-08-24']
[False  True]


`project_id` and `project_name` are unique.  

__1 Client can be in N Projects__

#### 2.5 Questions

In [941]:
#questions['question_id'] = questions['question_id'].astype(str)
#questions['campaign_id'] = questions['campaign_id'].astype(str)

exploration(questions)

First 10 rows of the table:


,question_id,campaign_id,campaign_code,question_code,question_name,question_type,question_category,question_order,question_is_highlighted,image_associated,created_at,updated_at_sys
0,1,1,CAM00001,Q00001,En caso de hacer una mejora en Strepsils/Stref...,M,2. Visibilidad,1.0,False,False,2025-09-30,2026-01-24
1,2,1,CAM00001,Q00002,"En caso de hacer una mejora en Nurofen, indícalo",M,2. Visibilidad,2.0,False,False,2025-09-30,2025-10-25
2,3,1,CAM00001,Q00003,Indica el nº de facings de Reflex Spray,S,2. Visibilidad,3.0,False,True,2025-09-30,2026-03-25
3,4,1,CAM00001,Q00004,¿Se ha realizado toda la instalación del vinil...,S,2.Visita,4.0,False,True,2025-09-30,2026-04-09
4,5,1,CAM00001,Q00005,Que producto en concreto?,T,1. Visita,5.0,False,False,2025-09-30,2025-11-01
5,6,1,CAM00001,Q00006,Indica el nº de facings de Nurofen,S,2. Visibilidad,6.0,True,False,2025-09-30,2026-03-03
6,7,1,CAM00001,Q00007,"En caso de hacer una mejora en Gaviscon, indícalo",M,2. Visibilidad,7.0,False,True,2025-09-30,2026-03-18
7,8,1,CAM00001,Q00008,Indica el nº de facings de Reflex Gel,S,2. Visibilidad,8.0,False,True,2025-09-30,2026-04-14
8,9,1,CAM00001,Q00009,Nombre de la persona formada,T,1. Visita,9.0,True,False,2025-09-30,2026-03-18
9,10,1,CAM00001,Q00010,Indica el nº de facings de Gaviscon,NaN,2. Visibilidad,10.0,False,False,2025-09-30,2025-12-29



Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 648 entries, 0 to 647
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   question_id              648 non-null    int64  
 1   campaign_id              648 non-null    int64  
 2   campaign_code            648 non-null    object 
 3   question_code            602 non-null    object 
 4   question_name            630 non-null    object 
 5   question_type            620 non-null    object 
 6   question_category        594 non-null    object 
 7   question_order           625 non-null    float64
 8   question_is_highlighted  648 non-null    bool   
 9   image_associated         648 non-null    bool   
 10  created_at               648 non-null    object 
 11  updated_at_sys           648 non-null    object 
dtypes: bool(2), float64(1), int64(2), object(7)
memory usage: 52.0+ KB


None


Descriptive statistics for all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
question_id,648.0,NaN,NaN,NaN,324.5,187.205769,1.0,162.75,324.5,486.25,648.0
campaign_id,648.0,NaN,NaN,NaN,37.524691,20.872751,1.0,20.0,38.0,56.0,72.0
campaign_code,648,72,CAM00033,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
question_code,602,602,Q00001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
question_name,630,26,CONTACTO VISITA,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN
question_type,620,4,S,287,NaN,NaN,NaN,NaN,NaN,NaN,NaN
question_category,594,4,1. Visita,264,NaN,NaN,NaN,NaN,NaN,NaN,NaN
question_order,625.0,NaN,NaN,NaN,5.544,3.361032,1.0,3.0,5.0,8.0,14.0
question_is_highlighted,648,2,False,486,NaN,NaN,NaN,NaN,NaN,NaN,NaN
image_associated,648,2,False,457,NaN,NaN,NaN,NaN,NaN,NaN,NaN



% of NaNs per column:
question_category          0.083333
question_code              0.070988
question_type              0.043210
question_order             0.035494
question_name              0.027778
question_id                0.000000
campaign_id                0.000000
campaign_code              0.000000
question_is_highlighted    0.000000
image_associated           0.000000
created_at                 0.000000
updated_at_sys             0.000000
dtype: float64

% of Empty Strings ('') per column:
campaign_code        0.0
question_code        0.0
question_name        0.0
question_type        0.0
question_category    0.0
created_at           0.0
updated_at_sys       0.0
dtype: float64

 Total duplicate rows: 0 (0.00%)

 % Outliers Detection (IQR Method):


,Outliers count,% Outliers,Min outlier found,Max outlier found
Column,,,,
question_id,0,0.00%,None,None
campaign_id,0,0.00%,None,None
question_order,0,0.00%,None,None



 Date summary:


,Min Date (Start),Max Date (End),Days Timespan,Valid Dates Count
Date Column,,,,
created_at,2025-09-11,2026-02-13,155,648
updated_at_sys,2025-10-01,2026-05-01,212,648


In [942]:
print(questions['created_at'].unique())
print(questions['updated_at_sys'].unique())
print(questions['question_is_highlighted'].unique())
print(questions['image_associated'].unique())

['2025-09-30' '2025-12-15' '2025-10-28' '2025-10-09' '2026-01-01'
 '2025-09-21' '2025-10-08' '2025-09-17' '2025-10-23' '2025-09-11'
 '2025-12-25' '2025-11-08' '2026-01-16' '2026-01-02' '2025-12-29'
 '2026-01-21' '2025-11-26' '2025-10-10' '2025-10-14' '2025-11-21'
 '2025-12-08' '2025-12-21' '2025-12-18' '2025-09-26' '2026-01-20'
 '2025-10-01' '2026-02-04' '2025-10-17' '2025-12-02' '2025-11-04'
 '2026-01-08' '2026-01-19' '2025-10-07' '2026-02-13' '2026-01-13'
 '2025-09-25' '2026-01-30' '2026-02-05' '2025-12-04' '2025-10-04'
 '2026-02-06' '2026-01-27' '2025-11-27' '2025-11-25' '2026-01-07'
 '2025-12-16' '2025-11-28' '2026-02-08' '2025-11-15' '2025-12-24'
 '2025-10-24' '2026-01-12' '2025-12-20' '2025-11-05' '2025-09-28']
['2026-01-24' '2025-10-25' '2026-03-25' '2026-04-09' '2025-11-01'
 '2026-03-03' '2026-03-18' '2026-04-14' '2025-12-29' '2025-11-06'
 '2026-01-16' '2026-03-02' '2026-03-05' '2026-01-19' '2026-03-20'
 '2025-11-22' '2025-12-23' '2026-01-10' '2026-02-25' '2025-10-30'
 '2026-01

`question_is` is unique.  

__1 Campaign can have N Questions__

#### 2.6 Responses

In [943]:
#responses['answer_id'] = responses['answer_id'].astype(str)
#responses['visit_id'] = responses['visit_id'].astype(str)
#responses['question_id'] = responses['question_id'].astype(str)
#responses["created_at"] = pd.to_datetime(responses["created_at"])
#responses["updated_at_sys"] = pd.to_datetime(responses["updated_at_sys"])

exploration(responses)

First 10 rows of the table:


,answer_id,visit_id,question_id,campaign_code,intervention_point_code,question_type,answer,expected_answer,created_at,updated_at_sys
0,1,1,4.0,CAM00001,PDV000056,S,"No, rechazan la instalación",Si,2025-12-31,2026-02-16
1,2,1,12.0,CAM00001,PDV000056,T,Sin comentarios,Si,2025-12-31,2026-02-16
2,3,1,8.0,CAM00001,PDV000056,S,"0, No tienen stock",Si,2025-12-31,2026-02-16
3,4,1,6.0,CAM00001,PDV000056,S,"0, No esta expuesto",No,2025-12-31,2026-02-16
4,5,1,5.0,CAM00001,PDV000056,T,Auxiliar de farmacia,No,2025-12-31,2026-02-16
5,6,1,3.0,CAM00001,PDV000056,S,4,NaN,2025-12-31,2026-02-16
6,7,1,9.0,CAM00001,PDV000056,T,Pendiente de confirmar,NaN,2025-12-31,2026-02-16
7,8,1,7.0,CAM00001,PDV000056,M,Sacar del cajon,No,2025-12-31,2026-02-16
8,9,1,11.0,CAM00001,PDV000056,S,2,NaN,2025-12-31,2026-02-16
9,10,1,13.0,CAM00001,PDV000056,T,NaN,NaN,2025-12-31,2026-02-16



Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10192 entries, 0 to 10191
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   answer_id                10192 non-null  int64  
 1   visit_id                 10192 non-null  int64  
 2   question_id              9904 non-null   float64
 3   campaign_code            10192 non-null  object 
 4   intervention_point_code  9611 non-null   object 
 5   question_type            9683 non-null   object 
 6   answer                   9167 non-null   object 
 7   expected_answer          3341 non-null   object 
 8   created_at               10192 non-null  object 
 9   updated_at_sys           10192 non-null  object 
dtypes: float64(1), int64(2), object(7)
memory usage: 796.4+ KB


None


Descriptive statistics for all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
answer_id,10192.0,NaN,NaN,NaN,5096.5,2942.321306,1.0,2548.75,5096.5,7644.25,10192.0
visit_id,10192.0,NaN,NaN,NaN,911.768838,513.534747,1.0,463.0,929.0,1367.0,1747.0
question_id,9904.0,NaN,NaN,NaN,336.295234,191.041516,1.0,164.0,368.0,502.0,648.0
campaign_code,10192,64,CAM00056,400,NaN,NaN,NaN,NaN,NaN,NaN,NaN
intervention_point_code,9611,248,PDV000084,105,NaN,NaN,NaN,NaN,NaN,NaN,NaN
question_type,9683,4,S,4317,NaN,NaN,NaN,NaN,NaN,NaN,NaN
answer,9167,128,Pendiente de confirmar,385,NaN,NaN,NaN,NaN,NaN,NaN,NaN
expected_answer,3341,2,Si,1678,NaN,NaN,NaN,NaN,NaN,NaN,NaN
created_at,10192,144,2026-02-13,153,NaN,NaN,NaN,NaN,NaN,NaN,NaN
updated_at_sys,10192,130,2026-05-19,219,NaN,NaN,NaN,NaN,NaN,NaN,NaN



% of NaNs per column:
expected_answer            0.672194
answer                     0.100569
intervention_point_code    0.057005
question_type              0.049941
question_id                0.028257
answer_id                  0.000000
visit_id                   0.000000
campaign_code              0.000000
created_at                 0.000000
updated_at_sys             0.000000
dtype: float64

% of Empty Strings ('') per column:
campaign_code              0.0
intervention_point_code    0.0
question_type              0.0
answer                     0.0
expected_answer            0.0
created_at                 0.0
updated_at_sys             0.0
dtype: float64

 Total duplicate rows: 0 (0.00%)

 % Outliers Detection (IQR Method):


,Outliers count,% Outliers,Min outlier found,Max outlier found
Column,,,,
answer_id,0,0.00%,None,None
visit_id,0,0.00%,None,None
question_id,0,0.00%,None,None



 Date summary:


,Min Date (Start),Max Date (End),Days Timespan,Valid Dates Count
Date Column,,,,
created_at,2025-12-27,2026-05-19,143,10192
updated_at_sys,2026-01-07,2026-05-19,132,10192


In [944]:
#print(responses['created_at'].unique())
#print(responses['updated_at_sys'].unique())
print(responses['expected_answer'].unique())

['Si' 'No' nan]


`answer_id` is unique.  
There are `answer_is` with null `question_id`.

__1 Visit can have N Responses__ and __1 Question can have N Responses__

#### 2.7 Route employee

In [945]:
#route_employee['route_employee_id'] = route_employee['route_employee_id'].astype(str)
#route_employee['route_id'] = route_employee['route_id'].astype(str)
#route_employee['employee_id'] = route_employee['employee_id'].astype(str)

exploration(route_employee)

First 10 rows of the table:


,route_employee_id,route_id,employee_id,main_employee,ip_percentage,created_at,updated_at,deleted_at
0,1,1,32,True,100.0,2025-12-26,2026-04-03,NaN
1,2,1,44,False,19.0,2025-12-26,2026-04-03,NaN
2,3,2,19,True,100.0,2026-01-15,2026-02-02,NaN
3,4,2,35,False,44.8,2026-01-15,2026-02-02,NaN
4,5,3,44,True,100.0,2026-03-29,2026-05-12,2026-04-14
5,6,3,9,False,19.4,2026-03-29,2026-05-12,NaN
6,7,4,28,True,100.0,2026-01-09,2026-01-22,NaN
7,8,4,16,False,35.4,2026-01-09,2026-01-22,NaN
8,9,5,36,True,100.0,2026-01-03,2026-04-26,NaN
9,10,5,35,False,34.3,2026-01-03,2026-04-26,NaN



Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 482 entries, 0 to 481
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   route_employee_id  482 non-null    int64  
 1   route_id           482 non-null    int64  
 2   employee_id        482 non-null    int64  
 3   main_employee      482 non-null    bool   
 4   ip_percentage      468 non-null    float64
 5   created_at         482 non-null    object 
 6   updated_at         482 non-null    object 
 7   deleted_at         25 non-null     object 
dtypes: bool(1), float64(1), int64(3), object(3)
memory usage: 27.0+ KB


None


Descriptive statistics for all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
route_employee_id,482.0,NaN,NaN,NaN,241.5,139.285678,1.0,121.25,241.5,361.75,482.0
route_id,482.0,NaN,NaN,NaN,121.0,69.642389,1.0,61.0,121.0,181.0,241.0
employee_id,482.0,NaN,NaN,NaN,23.599585,13.409721,1.0,13.0,24.0,35.0,45.0
main_employee,482,2,True,241,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ip_percentage,468.0,NaN,NaN,NaN,64.340171,36.49851,10.2,27.85,49.55,100.0,100.0
created_at,482,128,2026-01-02,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
updated_at,482,122,2026-04-12,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
deleted_at,25,24,2026-04-14,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN



% of NaNs per column:
deleted_at           0.948133
ip_percentage        0.029046
route_employee_id    0.000000
route_id             0.000000
employee_id          0.000000
main_employee        0.000000
created_at           0.000000
updated_at           0.000000
dtype: float64

% of Empty Strings ('') per column:
created_at    0.0
updated_at    0.0
deleted_at    0.0
dtype: float64

 Total duplicate rows: 0 (0.00%)

 % Outliers Detection (IQR Method):


,Outliers count,% Outliers,Min outlier found,Max outlier found
Column,,,,
route_employee_id,0,0.00%,None,None
route_id,0,0.00%,None,None
employee_id,0,0.00%,None,None
ip_percentage,0,0.00%,None,None



 Date summary:


,Min Date (Start),Max Date (End),Days Timespan,Valid Dates Count
Date Column,,,,
created_at,2025-10-02,2026-04-20,200,482
updated_at,2025-11-03,2026-05-19,197,482
deleted_at,2025-10-21,2026-07-29,281,25


In [946]:
#print(route_employee['created_at'].unique())
#print(route_employee['updated_at'].unique())
#print(route_employee['deleted_at'].unique())

In [947]:
route_employee_emp = route_employee[route_employee['employee_id'].notnull()]
display(route_employee_emp[['route_employee_id', 'route_id', 'employee_id']])

,route_employee_id,route_id,employee_id
0,1,1,32
1,2,1,44
2,3,2,19
3,4,2,35
4,5,3,44
...,...,...,...
477,478,239,9
478,479,240,4
479,480,240,33
480,481,241,24


`route_employee_id` is unique.  

__1 Route can have N Route Employees__, __1 Worker can have N Routes__ and __1 Worker can have N Routes Employee__

#### 2.8 Routes

In [948]:
#routes['route_id'] = routes['route_id'].astype(str)
#routes['campaign_id'] = routes['campaign_id'].astype(str)

exploration(routes)

First 10 rows of the table:


,route_id,route_code,route_name,campaign_id,campaign_code,route_start_date,route_end_date,route_status,delegation_code,recall_mail_sent,created_at,updated_at_sys
0,1,NaN,Ruta 1,1,CAM00001,2026-01-01,2026-03-01,NaN,DLG15,True,2025-12-26,2026-04-03
1,2,RUT00002,Ruta 2,1,CAM00001,2026-01-18,2026-04-10,NaN,DLG04,True,2026-01-15,2026-02-02
2,3,RUT00003,Ruta 3,1,CAM00001,2026-03-31,2026-05-13,Alta,DLG07,True,2026-03-29,2026-05-12
3,4,RUT00004,Ruta 4,2,CAM00002,2026-01-13,2026-02-19,Alta,DLG06,True,2026-01-09,2026-01-22
4,5,RUT00005,Ruta 5,2,CAM00002,2026-01-11,2026-02-23,Baja,DLG12,False,2026-01-03,2026-04-26
5,6,RUT00006,Ruta 6,2,CAM00002,2026-01-21,2026-02-25,Alta,DLG02,True,2026-01-20,2026-04-08
6,7,RUT00007,NaN,2,CAM00002,2026-01-16,2026-02-24,NaN,DLG11,False,2026-01-06,2026-03-21
7,8,RUT00008,Ruta 8,2,CAM00002,2026-01-20,2026-02-10,Pendiente,DLG09,True,2026-01-16,2026-03-12
8,9,RUT00009,Ruta 9,3,CAM00003,2025-11-21,2025-12-26,Alta,DLG08,False,2025-11-14,2026-02-26
9,10,RUT00010,Ruta 10,3,CAM00003,2025-11-28,2025-12-11,NaN,DLG05,True,2025-11-26,2026-02-28



Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241 entries, 0 to 240
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   route_id          241 non-null    int64 
 1   route_code        229 non-null    object
 2   route_name        233 non-null    object
 3   campaign_id       241 non-null    int64 
 4   campaign_code     241 non-null    object
 5   route_start_date  241 non-null    object
 6   route_end_date    241 non-null    object
 7   route_status      163 non-null    object
 8   delegation_code   214 non-null    object
 9   recall_mail_sent  241 non-null    bool  
 10  created_at        241 non-null    object
 11  updated_at_sys    241 non-null    object
dtypes: bool(1), int64(2), object(9)
memory usage: 21.1+ KB


None


Descriptive statistics for all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
route_id,241.0,NaN,NaN,NaN,121.0,69.714896,1.0,61.0,121.0,181.0,241.0
route_code,229,229,RUT00002,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_name,233,233,Ruta 1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
campaign_id,241.0,NaN,NaN,NaN,38.369295,21.717901,1.0,19.0,42.0,56.0,72.0
campaign_code,241,72,CAM00072,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_start_date,241,134,2025-11-20,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_end_date,241,168,2025-12-05,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_status,163,3,Pendiente,61,NaN,NaN,NaN,NaN,NaN,NaN,NaN
delegation_code,214,15,DLG10,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
recall_mail_sent,241,2,True,123,NaN,NaN,NaN,NaN,NaN,NaN,NaN



% of NaNs per column:
route_status        0.323651
delegation_code     0.112033
route_code          0.049793
route_name          0.033195
route_id            0.000000
campaign_id         0.000000
campaign_code       0.000000
route_start_date    0.000000
route_end_date      0.000000
recall_mail_sent    0.000000
created_at          0.000000
updated_at_sys      0.000000
dtype: float64

% of Empty Strings ('') per column:
route_code          0.0
route_name          0.0
campaign_code       0.0
route_start_date    0.0
route_end_date      0.0
route_status        0.0
delegation_code     0.0
created_at          0.0
updated_at_sys      0.0
dtype: float64

 Total duplicate rows: 0 (0.00%)

 % Outliers Detection (IQR Method):


,Outliers count,% Outliers,Min outlier found,Max outlier found
Column,,,,
route_id,0,0.00%,None,None
campaign_id,0,0.00%,None,None



 Date summary:


,Min Date (Start),Max Date (End),Days Timespan,Valid Dates Count
Date Column,,,,
route_start_date,2025-10-09,2026-04-30,203,241
route_end_date,2025-10-27,2026-10-17,355,241
created_at,2025-10-02,2026-04-20,200,241
updated_at_sys,2025-11-03,2026-05-19,197,241


In [949]:
#print(routes['created_at'].unique())
#print(routes['updated_at'].unique())

`route_id` is unique.  

__1 Campaign can have N Routes__

#### 2.9 Workers

In [950]:
#workers['employee_id'] = workers['employee_id'].astype(str)
#workers['company_id'] = workers['company_id'].astype(str)

exploration(workers)

First 10 rows of the table:


,employee_id,employee_first_name,employee_active_status,employee_hire_date,employee_address_province,employee_contract_type,company_id,created_at,updated_at_sys
0,1,David,True,2018-12-23,Castelló,NaN,1,2018-12-23,2025-06-14
1,2,Claudia,True,2021-07-02,Guarda,PART,1,2021-07-02,2023-12-22
2,3,Laura,True,2019-04-27,Canillo,INDEF,1,2019-04-27,2025-05-27
3,4,Elena,True,2021-12-05,Castellon,TEMP,1,2021-12-05,2022-04-29
4,5,Pablo,False,2021-06-29,MÁLAGA,INDEF,1,2021-06-29,2023-08-23
5,6,Carlos,False,2018-08-09,TARRAGONA,NaN,1,2018-08-09,2022-08-04
6,7,David,True,2021-07-03,Santarém,TEMP,1,2021-07-03,2026-01-22
7,8,Carlos,True,2023-04-03,Castellón,NaN,1,2023-04-03,2024-07-20
8,9,María,True,2020-07-05,Cuenca,PART,1,2020-07-05,2025-08-04
9,10,Ana,False,2023-06-20,ASTURIAS,TEMP,1,2023-06-20,2025-06-02



Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   employee_id                47 non-null     int64 
 1   employee_first_name        47 non-null     object
 2   employee_active_status     44 non-null     object
 3   employee_hire_date         45 non-null     object
 4   employee_address_province  46 non-null     object
 5   employee_contract_type     33 non-null     object
 6   company_id                 47 non-null     int64 
 7   created_at                 47 non-null     object
 8   updated_at_sys             47 non-null     object
dtypes: int64(2), object(7)
memory usage: 3.4+ KB


None


Descriptive statistics for all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
employee_id,47.0,NaN,NaN,NaN,22.382979,13.191818,1.0,11.0,22.0,33.5,45.0
employee_first_name,47,14,Pablo,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employee_active_status,44,2,True,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employee_hire_date,45,43,2018-08-09,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employee_address_province,46,33,Castellon,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employee_contract_type,33,3,TEMP,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
company_id,47.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
created_at,47,45,2018-08-09,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
updated_at_sys,47,45,2022-08-04,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN



% of NaNs per column:
employee_contract_type       0.297872
employee_active_status       0.063830
employee_hire_date           0.042553
employee_address_province    0.021277
employee_id                  0.000000
employee_first_name          0.000000
company_id                   0.000000
created_at                   0.000000
updated_at_sys               0.000000
dtype: float64

% of Empty Strings ('') per column:
employee_first_name          0.0
employee_active_status       0.0
employee_hire_date           0.0
employee_address_province    0.0
employee_contract_type       0.0
created_at                   0.0
updated_at_sys               0.0
dtype: float64

 Total duplicate rows: 2 (4.26%)

 % Outliers Detection (IQR Method):


,Outliers count,% Outliers,Min outlier found,Max outlier found
Column,,,,
employee_id,0,0.00%,None,None
company_id,0,0.00%,None,None



 Date summary:


,Min Date (Start),Max Date (End),Days Timespan,Valid Dates Count
Date Column,,,,
employee_hire_date,2018-05-08,2025-05-23,2572,45
created_at,2018-05-08,2025-05-23,2572,47
updated_at_sys,2021-03-07,2026-04-19,1869,47


We can see there is not any unique column. Let's confirm if there are duplicates.

In [951]:
display(workers[workers.duplicated(keep=False)].sort_values(by='employee_id'))

,employee_id,employee_first_name,employee_active_status,employee_hire_date,employee_address_province,employee_contract_type,company_id,created_at,updated_at_sys
5,6,Carlos,False,2018-08-09,TARRAGONA,NaN,1,2018-08-09,2022-08-04
45,6,Carlos,False,2018-08-09,TARRAGONA,NaN,1,2018-08-09,2022-08-04
10,11,Pablo,True,2024-10-11,Hauts-de-Seine,PART,1,2024-10-11,2024-12-22
46,11,Pablo,True,2024-10-11,Hauts-de-Seine,PART,1,2024-10-11,2024-12-22


In [952]:
print(workers['employee_address_province'].unique())

['Castelló' 'Guarda' 'Canillo' 'Castellon' 'MÁLAGA' 'TARRAGONA' 'Santarém'
 'Castellón' 'Cuenca' 'ASTURIAS' 'Hauts-de-Seine' 'Rioja, La' 'BADAJOZ'
 'Pontevedra' 'LUGO' nan 'SANTA CRUZ DE TENERIFE' 'VALLADOLID'
 'Escaldes-Engordany' 'Portalegre' 'Viseu Dão Lafões' 'Alpes (Hautes)'
 'Viseu' 'CIUDAD REAL' 'MADRID' 'Granada' 'ZAMORA' 'Almería' 'A Coruña'
 'PONTEVEDRA' 'Madrid' 'Ávila' 'Barcelona' 'Palmas, Las']


In [953]:
#print(workers['created_at'].unique())
#print(workers['updated_at'].unique())

`employee_id` is unique (without taking into account the duplicate).

#### 2.10 Visits

In [954]:
#visits['visit_id'] = visits['visit_id'].astype(str)
#visits['campaign_id'] = visits['campaign_id'].astype(str)
#visits['intervention_point_id'] = visits['intervention_point_id'].astype(str)
#visits['route_id'] = visits['route_id'].astype(str)

exploration(visits)

First 10 rows of the table:


,visit_id,campaign_id,campaign_code,project_code,intervention_point_id,intervention_point_code,route_id,route_code,visit_date,visit_time,visit_status,visit_type,is_client_billable,created_at,updated_at_sys
0,1,1,CAM00001,PRJ0001,56,PDV000056,3.0,RUT00003,2026-01-02,18:15:00,INFO,NaN,True,2025-12-31,2026-02-16
1,2,1,CAM00001,PRJ0001,36,PDV000036,3.0,RUT00003,2026-02-05,12:30:00,INCID,NaN,False,2026-02-03,2026-05-14
2,3,1,CAM00001,PRJ0001,116,PDV000116,3.0,RUT00003,2026-03-08,17:15:00,INCID,C,True,2026-03-04,2026-05-17
3,4,1,CAM00001,PRJ0001,24,PDV000024,3.0,NaN,2026-02-04,16:15:00,INCID,C,False,2026-01-30,2026-04-20
4,5,1,CAM00001,PRJ0001,83,NaN,3.0,RUT00003,2026-01-05,13:30:00,NOVIS,C,True,2026-01-01,2026-02-21
5,6,1,CAM00001,PRJ0001,113,PDV000113,3.0,RUT00003,2026-01-03,11:30:00,INCID,NaN,True,2026-01-03,2026-03-03
6,7,1,CAM00001,PRJ0001,28,PDV000028,2.0,RUT00002,2026-03-06,10:15:00,NOVIS,V,True,2026-03-01,2026-04-17
7,8,1,CAM00001,PRJ0001,249,PDV000249,1.0,RUT00001,2026-01-07,15:00:00,INCID,C,True,2026-01-06,2026-02-28
8,9,1,CAM00001,PRJ0001,24,PDV000024,3.0,NaN,2026-02-25,10:30:00,INCID,NaN,True,2026-02-20,2026-04-14
9,10,1,CAM00001,PRJ0001,181,PDV000181,1.0,RUT00001,2026-03-01,14:30:00,INFO,NaN,True,2026-03-01,2026-04-14



Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1762 entries, 0 to 1761
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   visit_id                 1762 non-null   int64  
 1   campaign_id              1762 non-null   int64  
 2   campaign_code            1762 non-null   object 
 3   project_code             1762 non-null   object 
 4   intervention_point_id    1762 non-null   int64  
 5   intervention_point_code  1667 non-null   object 
 6   route_id                 1606 non-null   float64
 7   route_code               1628 non-null   object 
 8   visit_date               1725 non-null   object 
 9   visit_time               1582 non-null   object 
 10  visit_status             1691 non-null   object 
 11  visit_type               1096 non-null   object 
 12  is_client_billable       1655 non-null   object 
 13  created_at               1762 non-null   obje

None


Descriptive statistics for all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
visit_id,1762.0,NaN,NaN,NaN,868.772985,505.581578,1.0,427.25,866.5,1306.75,1747.0
campaign_id,1762.0,NaN,NaN,NaN,37.102724,21.08341,1.0,18.0,37.0,56.0,72.0
campaign_code,1762,64,CAM00066,48,NaN,NaN,NaN,NaN,NaN,NaN,NaN
project_code,1762,32,PRJ0020,151,NaN,NaN,NaN,NaN,NaN,NaN,NaN
intervention_point_id,1762.0,NaN,NaN,NaN,126.990352,72.925735,1.0,65.0,129.0,190.0,250.0
intervention_point_code,1667,248,PDV000190,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_id,1606.0,NaN,NaN,NaN,4239.113325,19324.982974,1.0,60.0,120.0,183.0,99970.0
route_code,1628,270,RUT00142,34,NaN,NaN,NaN,NaN,NaN,NaN,NaN
visit_date,1725,242,2026-03-06,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
visit_time,1582,48,09:00:00,48,NaN,NaN,NaN,NaN,NaN,NaN,NaN



% of NaNs per column:
visit_type                 0.377980
visit_time                 0.102157
route_id                   0.088536
route_code                 0.076050
is_client_billable         0.060726
intervention_point_code    0.053916
visit_status               0.040295
visit_date                 0.020999
visit_id                   0.000000
campaign_id                0.000000
campaign_code              0.000000
project_code               0.000000
intervention_point_id      0.000000
created_at                 0.000000
updated_at_sys             0.000000
dtype: float64

% of Empty Strings ('') per column:
campaign_code              0.0
project_code               0.0
intervention_point_code    0.0
route_code                 0.0
visit_date                 0.0
visit_time                 0.0
visit_status               0.0
visit_type                 0.0
is_client_billable         0.0
created_at                 0.0
updated_at_sys             0.0
dtype: float64

 Total duplicate rows: 15 (0

,Outliers count,% Outliers,Min outlier found,Max outlier found
Column,,,,
visit_id,0,0.00%,NaN,NaN
campaign_id,0,0.00%,NaN,NaN
intervention_point_id,0,0.00%,NaN,NaN
route_id,70,3.97%,90052.0,99970.0



 Date summary:


,Min Date (Start),Max Date (End),Days Timespan,Valid Dates Count
Date Column,,,,
visit_date,2026-01-01,2026-05-19,138,1486
created_at,2025-12-27,2026-05-19,143,1762
updated_at_sys,2026-01-07,2026-05-19,132,1762


We can see there is not any unique column. Let's confirm if there are duplicates.

In [955]:
display(visits[visits['visit_id'].duplicated(keep=False)].sort_values(by='visit_id'))

,visit_id,campaign_id,campaign_code,project_code,intervention_point_id,intervention_point_code,route_id,route_code,visit_date,visit_time,visit_status,visit_type,is_client_billable,created_at,updated_at_sys
19,20,2,CAM00002,PRJ0001,195,PDV000195,7.0,RUT00007,2026-01-24,08:45:00,INCID,T,NaN,2026-01-21,2026-04-18
1749,20,2,CAM00002,PRJ0001,195,PDV000195,7.0,RUT00007,2026-01-24,08:45:00,INCID,T,NaN,2026-01-21,2026-04-18
123,124,5,CAM00005,PRJ0003,147,NaN,17.0,NaN,2026-02-04,13:00:00,INCID,C,True,2026-02-04,2026-02-11
1758,124,5,CAM00005,PRJ0003,147,NaN,17.0,NaN,2026-02-04,13:00:00,INCID,C,True,2026-02-04,2026-02-11
157,158,7,CAM00007,PRJ0005,134,PDV000134,21.0,RUT00021,2026-01-06,11:30:00,INCID,C,True,2026-01-01,2026-04-16
1756,158,7,CAM00007,PRJ0005,134,PDV000134,21.0,RUT00021,2026-01-06,11:30:00,INCID,C,True,2026-01-01,2026-04-16
158,159,7,CAM00007,PRJ0005,148,PDV000148,23.0,RUT00023,2026-01-29,15:15:00,NOVIS,T,True,2026-01-28,2026-02-07
1753,159,7,CAM00007,PRJ0005,148,PDV000148,23.0,RUT00023,2026-01-29,15:15:00,NOVIS,T,True,2026-01-28,2026-02-07
170,171,7,CAM00007,PRJ0005,226,PDV000226,22.0,RUT00022,2026-02-08,15:45:00,INCID,NaN,False,2026-02-04,2026-02-17
1759,171,7,CAM00007,PRJ0005,226,PDV000226,22.0,RUT00022,2026-02-08,15:45:00,INCID,NaN,False,2026-02-04,2026-02-17


In [956]:
print(visits['visit_status'].unique())

['INFO' 'INCID' 'NOVIS' 'OK' nan]


`visit_id` is unique (without taking into account the dupplicates).  
There are instances of `visit_id` records that lack an assigned `route_id` (null values). This indicates the presence of "orphan" visits that are not currently linked to any scheduled field route.

__1 Point of Sale can have N visits__, __1 Campaign can have N visits__ and __1 Route can have N visits__

# 3. Data Quality Issues

In the previous point, we already have seen some quality issues. Three of them are:

##### 1. Duplicated rows in Visits and Workers tables
There are completely identical row duplicates within the `clients`, `visits` and `workers` tables. I would handle this issue by applying a deduplication process to remove these perfect clones, keeping only the first occurrence. This approach is justified because duplicate rows do not add any new analytical information; instead, they skew statistical metrics, artificially inflate counts, and cause unnecessary memory overhead during data processing.

##### 2. Categorical Inconsistency in Point of Sale Provinces
The exploratory analysis of the `pos` table revealed heavy text dispersion in the `intervention_point_province` column. Identical regions are split into multiple categories due to inconsistent capitalization (`ZAMORA` vs. `Zamora`), grammatical inversions (`La Rioja` vs. `Rioja, La`), and regional accent variations (`Castellón` vs. `Castellon`). I will handle this by implementing strict text-cleaning routines (forcing lowercasing and standard title casing) using a macro in the staging layer. Standardizing this is critical because when this dimension is joined to the main facts, these inconsistencies propagate downstream, splitting identical operational areas into separate categories, which breaks geographical BI filters and ruins dashboard map visualizations.

##### 3. Inconsistent data formats in dates and countries
There is a clear lack of standardization across several core columns. For instance, `campaign_start_date` and `campaign_end_date` mix `DD/MM/YYYY` and `YYYY-MM-DD` formats, while the `country` column in the `clients` table mixes Spanish and English names (such as `['spain', 'España', 'SPAIN']`). I will handle this by implementing data normalization, parsing all dates into a unified datetime format and mapping country names to a single language criteria. 

# 4. Star Schema Proposal

The most granular event available in the dataset is a response given to a specific question during a specific visit. Therefore, the fact table will be modeled at this level of detail. This grain was chosen because it is the lowest level of detail available in the source data and allows flexible analysis of visits, campaigns, questions, field workers, routes, and intervention points.

#### Schema diagram


![Star Schema](star_schema.png)

This design follows a star schema approach, with a single central fact table connected directly to all analytical dimensions. The chosen grain (one response per question per visit) preserves maximum detail and enables aggregation at visit, campaign, project, client, worker, route, or intervention point level without losing information. It is represented by `answer_id`.

The dimensional layout is structured following standard Kimball methodology, aiming to optimize query performance while keeping the data repository DRY (*Don't Repeat Yourself*). Each dimension has been chosen and designed to serve a specific analytical purpose:

* **Campaign, Project, and Client Consolidation (`DIM_CAMPAIGN`):** To avoid a complex Snowflake schema that degrades query performance, we have flattened the company hierarchy directly into this dimension. Since clients and projects naturally roll up into a campaign in a linear manner, merging them allows us to maintain rich descriptive attributes without splitting them into separate tables.  

* **The Route-Worker Relationship via Bridge Modeling (`DIM_ROUTE`, `DIM_WORKERS` & `BRIDGE_ROUTE_EMPLOYEE`):** According to the business rules, a field route can be assigned to multiple workers simultaneously. Leaving the employee identifier directly in the storage-level fact table would break the atomic grain and cause artificial row inflation. To solve this, we decoupled workers into `DIM_WORKERS` and managed the many-to-many relationship through `BRIDGE_ROUTE_EMPLOYEE`.  

* **`DIM_QUESTION`:** This dimension isolates the static metadata of the evaluation forms. By decoupling fields like `question_name`, `question_type`, and `question_category` from the core fact table, we drastically reduce storage overhead while allowing immediate reporting categorization by question typology.  

* **`DIM_VISIT`:** The visit dimension acts as the operational wrapper of the execution event, capturing high-level transaction states such as `visit_status`. Keeping this decoupled ensures that transactional attributes can be updated or tracked without modifying the underlying historical survey answers.  

* **(`DIM_POS`):** Point-of-sale dimensions are isolated to allow immediate regional segmentation, such as evaluating field execution and campaign performance by province or specific location names.  

* **`DIM_DATE`:** Moving beyond raw dates into a structured calendar enables complex time-intelligence operations, allowing the business to analyze operational trends by quarter, month, or day of the week.